In [1]:
import simulator_gpu as simg

In [3]:
import numpy as np
import simulator as sim
from NoisyCircuits.utils.BuildQubitGateModel import BuildModel
from NoisyCircuits.utils.CreateNoiseModel import CreateNoiseModel
from NoisyCircuits import QuantumCircuit as nqc
from pympler import asizeof

2026-03-31 18:41:38,177	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:
simg.get_gpu_count()

4

In [5]:
file_path = "https://raw.githubusercontent.com/Sats2/NoisyCircuits/main/noise_models/Sample_Noise_Model_Heron_QPU.csv"
noise_model = CreateNoiseModel(calibration_data_file=file_path, 
                                basis_gates=[["x", "sx", "rz", "rx"], ["cz", "rzz"]]).create_noise_model()

In [6]:
model_outputs = BuildModel(noise_model, 2, 2, 1e-5, [["sx", "x", "rx", "rz"], ["cz", "rzz"]], False).build_qubit_gate_model()

In [7]:
single, double, measure, connect = model_outputs

In [8]:
print("Size of single qubit gate model:", asizeof.asizeof(single) / 1024**2, "MB")
print("Size of two qubit gate model:", asizeof.asizeof(double) / 1024**2, "MB")

Size of single qubit gate model: 0.07619476318359375 MB
Size of two qubit gate model: 0.0619964599609375 MB


In [9]:
single_qubit_instructions = {}
for qubit in single.keys():
    single_qubit_instructions[qubit] = {}
    for gate in single[qubit].keys():
        single_qubit_instructions[qubit][gate] = single[qubit][gate]["qubit_channel"]

In [10]:
two_qubit_instructions = {}
for gate in double.keys():
    two_qubit_instructions[gate] = {}
    for qpair in double[gate].keys():
        two_qubit_instructions[gate][qpair] = double[gate][qpair]["qubit_channel"]

In [11]:
print("Testing Reduced Model:")
print("Single Qubit Instructions: ", asizeof.asizeof(single_qubit_instructions) / 1024**2, "MB")
print("Two Qubit Instructions: ", asizeof.asizeof(two_qubit_instructions) / 1024**2, "MB")

Testing Reduced Model:
Single Qubit Instructions:  0.02146148681640625 MB
Two Qubit Instructions:  0.0203094482421875 MB


In [12]:
def apply_cnot(instructions, q1,q2):
    instructions.append(["rz", [q2, q2], np.pi/2])
    instructions.append(["sx", [q2, q2], 0])
    instructions.append(["rz", [q2, q2], np.pi/2])
    instructions.append(["cz", [q1, q2], 0])
    instructions.append(["rz", [q2, q2], np.pi/2])
    instructions.append(["sx", [q2, q2], 0])
    instructions.append(["rz", [q2, q2], np.pi/2])
    instructions.append(["rx", [q1, q1], -np.pi])
    instructions.append(["x", [q1, q1], 0])

In [14]:
instructions = [
    ["sx", [0, 0], 0],
    ["rz", [0, 0], np.pi/2],
    ["sx", [0, 0], 0],
]
for i in range(23):
    apply_cnot(instructions, i, i+1)

In [9]:
circ = nqc(16, noise_model, "heron", 100, 10, "qulacs", False, False, 1e-5)

Successfully switched backend to qulacs.


2026-03-30 16:54:12,124	INFO worker.py:2007 -- Started a local Ray instance.
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [10]:
circ.refresh()
circ.H(0)
for i in range(15):
    circ.CX(i, i+1)

In [ ]:
pure_state = circ.run_pure_state(list(range(16)))
density_matrix = circ.run_with_density_matrix(list(range(16)))
density_approx = circ.execute(list(range(16)), 1000)

In [12]:
fid = np.sum(np.sqrt(density_matrix * density_approx))**2
assert np.isclose(np.sum(density_approx), 1.0, atol=1e-6)
print("Fidelity between density matrix and approximation:", fid)

Fidelity between density matrix and approximation: 0.9992919534532095


In [13]:
m = circ.measurement_error_operator

In [16]:
num_qubits = 16
num_traj = 1000
num_cores = 20
noise = False
state = np.zeros(2**num_qubits, dtype=np.complex128)
sim.simulate_circuit(instructions, state, single_qubit_instructions, two_qubit_instructions, num_qubits, noise, num_traj, num_cores)
state = np.abs(state)**2
# fid_pure_state = np.sum(np.sqrt(pure_state * state))**2

num_qubits = 16
num_traj = 1000
num_cores = 50
noise = True
probs_cpp_approx = np.zeros(2**num_qubits, dtype=np.complex128)
sim.simulate_circuit(instructions, probs_cpp_approx, single_qubit_instructions, two_qubit_instructions, num_qubits, noise, num_traj, num_cores)
if noise:
    probs_cpp_approx = probs_cpp_approx / num_traj
# probs_cpp_approx = np.abs(state)**2
# probs_cpp_approx = np.dot(m, probs_cpp_approx)
# density_matrix_no_measure = np.linalg.solve(m, density_matrix)
assert np.isclose(np.sum(probs_cpp_approx), 1.0, atol=1e-6)
# fid_cpp_approx = np.sum(np.sqrt(probs_cpp_approx * density_matrix))**2

# print("Fidelity between pure state and approximation:", fid_pure_state)
# print("Fidelity between density matrix and approximation:", fid_cpp_approx)

In [ ]:
#TODO: Test GPU simulator

In [18]:
num_qubits = 25
num_traj = 1000
num_cores = 20
noise = False
state = np.zeros(2**num_qubits, dtype=np.complex128)
simg.simulate_circuit(instructions, state, single_qubit_instructions, two_qubit_instructions, num_qubits, noise, num_traj)
state = np.abs(state)**2
# fid_pure_state = np.sum(np.sqrt(pure_state * state))**2

# num_qubits = 16
# num_traj = 1000
# num_cores = 50
# noise = True
# probs_cpp_approx = np.zeros(2**num_qubits, dtype=np.complex128)
# simg.simulate_circuit(instructions, probs_cpp_approx, single_qubit_instructions, two_qubit_instructions, num_qubits, noise, num_traj)
# if noise:
#     probs_cpp_approx = probs_cpp_approx / num_traj
# # probs_cpp_approx = np.abs(state)**2
# # probs_cpp_approx = np.dot(m, probs_cpp_approx)
# # density_matrix_no_measure = np.linalg.solve(m, density_matrix)
# assert np.isclose(np.sum(probs_cpp_approx), 1.0, atol=1e-6)
# fid_cpp_approx = np.sum(np.sqrt(probs_cpp_approx * density_matrix))**2

# print("Fidelity between pure state and approximation:", fid_pure_state)
# print("Fidelity between density matrix and approximation:", fid_cpp_approx)

In [19]:
state

array([1., 0., 0., ..., 0., 0., 0.], shape=(33554432,))

In [25]:
circ.shutdown()